<a href="https://colab.research.google.com/github/DaniilDonskoy/building_maintenance_agents/blob/feature%2Fincident-data-analysis/research/%D0%90%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_%D0%B7%D0%B0%D1%8F%D0%B2%D0%BE%D0%BA_%D0%BC%D0%B0%D1%80%D1%82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

from natasha import Segmenter, MorphVocab, NewsEmbedding, NewsMorphTagger, Doc
import re

root = Path.cwd()
if not (root / "incident_requests").exists() and (root.parent / "incident_requests").exists():
    root = root.parent

sys.path.append(str(root))

from incident_requests.incident_requests import IncidentRequestsPreprocessor

pd.set_option("display.max_colwidth", None)


In [3]:
sheet_id = "1ya7OgCwUC3a_BylVxideUUAe5I18Pbr4"
sheet_name = "Sheet1"

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"

df = pd.read_csv(url)

print(df.columns)
display(df.head(1))

Index(['Журнал заявок\nООО "Эксплуатация Главстрой-СПб"\nОтчёт сформирован 16.03.2026 09:17 Дата',
       'Время', 'Источник', 'Адрес', 'Пом.', 'Категория', 'Подкатегория',
       'Описание', 'Комментарий к выполненным работам', 'Статус заявки',
       'Желаемое время выполнения', 'Дата исполнения', 'Исполнители',
       'Координаторы', 'Перечень материалов', 'Услуги', 'Стоимость',
       'Вложения'],
      dtype='str')


,"Журнал заявок\nООО ""Эксплуатация Главстрой-СПб""\nОтчёт сформирован 16.03.2026 09:17 Дата",Время,Источник,Адрес,Пом.,Категория,Подкатегория,Описание,Комментарий к выполненным работам,Статус заявки,Желаемое время выполнения,Дата исполнения,Исполнители,Координаторы,Перечень материалов,Услуги,Стоимость,Вложения
0,15.03.2026,23:19,Диспетчер,"г Санкт-Петербург, п Парголово, ул Николая Рубцова, д. 5 стр. 1",309,Аварийная,Протечка,"СИЛЬНАЯ ТЕЧЬ СТОЯКА ГВС В ВАННОЙ, ОТКЛ. СТ. 29 ГВС В/З, ТРЕБ. ЗАМЕНА ОТСЕЧНОГО КРАНА, ВЫРВАЛО ШТОК\nГИЛЬДИЯ: Заменили два крана 1/2 на хгвс ст. 29 в/з\nЗапустили развоздушили",NaN,Принята к исполнению,с 09:00 16.03.2026 по 13:00 16.03.2026,NaN,NaN,"ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА, Бригадир санитарно-технических работ Журавский Антон Петрович",NaN,NaN,0,Нет


# Предпросмотр

In [4]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 458 entries, 0 to 457
Data columns (total 18 columns):
 #   Column                                                                                  Non-Null Count  Dtype  
---  ------                                                                                  --------------  -----  
 0   Журнал заявок
ООО "Эксплуатация Главстрой-СПб"
Отчёт сформирован 16.03.2026 09:17 Дата  458 non-null    str    
 1   Время                                                                                   458 non-null    str    
 2   Источник                                                                                458 non-null    str    
 3   Адрес                                                                                   458 non-null    str    
 4   Пом.                                                                                    176 non-null    str    
 5   Категория                                                                         

In [5]:
display(df['Пом.'].value_counts())
display(df['Пом.'].unique()) #Пом. - это квартира? и что означают буквы в номере

Пом.
267     3
450     3
141     3
919     2
26      2
       ..
1064    1
680     1
708     1
739     1
611     1
Name: count, Length: 157, dtype: int64

<StringArray>
[  '309',   '267',   '575',     nan,    '21',     '1',   '742',   '555',
    '90',   '168',
 ...
    '9Н', '2129П',   '65Н',    '36',  '1054',  '1064',   '680',   '708',
   '739',   '611']
Length: 158, dtype: str

In [6]:
df['Источник'].value_counts() # житель = из обращения?

Источник
Диспетчер       415
Житель           35
Из обращения      8
Name: count, dtype: int64

In [7]:
df['Категория'].value_counts() #бесполезный признак

Категория
Аварийная    458
Name: count, dtype: int64

In [8]:
df['Подкатегория'].value_counts()

Подкатегория
Лифт                214
Протечка            193
Электроснабжение     49
Прочие работы         2
Name: count, dtype: int64

In [9]:
df[df['Подкатегория'] == 'Прочие работы']

# ТЕЧЬ РАДИАТОРА В КОМНАТЕ попала в 'Прочие работы' -> тип заявки лучше узнавать из описания

,"Журнал заявок\nООО ""Эксплуатация Главстрой-СПб""\nОтчёт сформирован 16.03.2026 09:17 Дата",Время,Источник,Адрес,Пом.,Категория,Подкатегория,Описание,Комментарий к выполненным работам,Статус заявки,Желаемое время выполнения,Дата исполнения,Исполнители,Координаторы,Перечень материалов,Услуги,Стоимость,Вложения
195,09.03.2026,11:18,Диспетчер,"г Санкт-Петербург, п Парголово, ул Михаила Дудина, д. 25 корп. 1 лит. А",64Н,Аварийная,Прочие работы,"КП, СТУДИЯ КРАСОТЫ ""ЛАЗЕР ПРО ЛАБ""- ВЫТЕКАНИЕ ИЗ УНИТАЗА 89811614344\nГИЛЬДИЯ6Промыли лежак канализации в подвале кипятком 12 метров. требуется убрать воду в подвале с 5 парадной по 4ю",NaN,Принята к исполнению,с 10:00 10.03.2026 по 12:00 10.03.2026,NaN,NaN,ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА,NaN,NaN,0,Нет
298,05.03.2026,17:37,Диспетчер,"г Санкт-Петербург, п Парголово, ул Михаила Дудина, д. 25 корп. 2 лит. А",1143,Аварийная,Прочие работы,ТЕЧЬ РАДИАТОРА В КОМНАТЕ 89312870024\n20:26 Пребрали соединение батареи радиатора,NaN,Принята к исполнению,с 17:45 05.03.2026 по 18:00 05.03.2026,NaN,NaN,"ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА, Главный инженер Софронов Андрей Иванович",NaN,NaN,0,Нет


In [10]:
df['Комментарий к выполненным работам'].isna().sum()

np.int64(458)

In [11]:
df['Статус заявки'].value_counts()

Статус заявки
Принята к исполнению    458
Name: count, dtype: int64

In [12]:
df['Дата исполнения'].isna().sum() #мб полезный признак, узнаем время на работу

np.int64(458)

In [13]:
df['Исполнители'].isna().sum() # полезно, чтобы узнать количество задействованных людей

np.int64(458)

In [14]:
df['Координаторы'].value_counts() #заполнено, но непонятно как использовать

Координаторы
Спецтрест 27 – ЛИФТ Шкапцов А Л, Ведущий инженер по подъемно-транспортному оборудованию Гореликов Павел Вячеславович, Инженер по подъемно-транспортному оборудованию Горюнов Сергей Александрович                                                           91
ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА                                                                                                                                                                                                                     84
Ведущий инженер по подъемно-транспортному оборудованию Гореликов Павел Вячеславович, Инженер по подъемно-транспортному оборудованию Горюнов Сергей Александрович, Спецтрест 27 – ЛИФТ Шкапцов А Л                                                           62
ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА, Главный инженер Софронов Андрей Иванович                                                                                                                                             

In [15]:
df['Перечень материалов'].isna().sum()

np.int64(458)

In [16]:
df['Услуги'].isna().sum()

np.int64(458)

In [17]:
df['Стоимость'].value_counts()

Стоимость
0    458
Name: count, dtype: int64

In [18]:
df['Вложения'].value_counts() # Что здесь имеется в виду? что означает да/нет?

Вложения
Нет    434
Да      24
Name: count, dtype: int64

# Парсинг инцидентов

In [19]:
processor = IncidentRequestsPreprocessor(df)

processed_df = processor.preprocess()
display(processed_df.head())

/home/fyodorg/Projects/building_maintenance_agents/.venv/lib/python3.12/site-packages/pymorphy2/analyzer.py:114: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


,Дата,Время,Адрес,Пом.,Подкатегория,Описание,Комментарий к выполненным работам,Статус заявки,Желаемое время выполнения,Дата исполнения,Исполнители,Координаторы,Перечень материалов,Услуги,Стоимость,Вложения,Тип инцидента
0,15.03.2026,23:19,"г Санкт-Петербург, п Парголово, ул Николая Рубцова, д. 5 стр. 1",309,Протечка,"СИЛЬНАЯ ТЕЧЬ СТОЯКА ГВС В ВАННОЙ, ОТКЛ. СТ. 29 ГВС В/З, ТРЕБ. ЗАМЕНА ОТСЕЧНОГО КРАНА, ВЫРВАЛО ШТОК\nГИЛЬДИЯ: Заменили два крана 1/2 на хгвс ст. 29 в/з\nЗапустили развоздушили",NaN,Принята к исполнению,с 09:00 16.03.2026 по 13:00 16.03.2026,NaN,NaN,"ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА, Бригадир санитарно-технических работ Журавский Антон Петрович",NaN,NaN,0,Нет,Утечка стояка
1,15.03.2026,21:32,"г Санкт-Петербург, пр-кт Гладышевский, д. 38 корп. 2 стр. 1",267,Протечка,течь с потолка-ТЕЧЬ С КРОВЛИ,NaN,Принята к исполнению,с 09:00 16.03.2026 по 15:00 16.03.2026,NaN,NaN,Начальник отделения Григорьев Игорь Валерьевич,NaN,NaN,0,Да,Прочее
2,15.03.2026,20:48,"г Санкт-Петербург, п Парголово, ул Фёдора Абрамова, д. 21 корп. 1 лит. А",575,Протечка,"ТЕЧЬ ПО СТЕНЕ В КОМНАТЕ +ТЕЧЕТ В КВАРТИРНОМ КОРИДОРЕ ,ТЕЧЬ СПУСКНИКА НА ТЕХ/ЭТ ,ЗАКРЫЛИ СПУСКНИК",NaN,Принята к исполнению,с 21:00 15.03.2026 по 21:15 15.03.2026,NaN,NaN,"Главный инженер Софронов Андрей Иванович, ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА",NaN,NaN,0,Нет,Прочее
3,15.03.2026,20:29,"г Санкт-Петербург, п Парголово, ул Фёдора Абрамова, д. 16 корп. 1 лит. А",NaN,Лифт,3 пар. пас. 1934 застревание,NaN,Принята к исполнению,с 20:30 15.03.2026 по 19:00 16.03.2026,NaN,NaN,"Ведущий инженер по подъемно-транспортному оборудованию Гореликов Павел Вячеславович, Инженер по подъемно-транспортному оборудованию Горюнов Сергей Александрович, Спецтрест 27 – ЛИФТ Шкапцов А Л",NaN,NaN,0,Нет,Прочее
4,15.03.2026,19:47,"г Санкт-Петербург, п Парголово, ул Валерия Гаврилина, д. 3 корп. 1 лит. А",21,Протечка,"течь п/суш. (981) 822-05-22, течь соед. на ст. гвс н/з, откл. ст. гвс 18, ТРЕБ. РАЗБОР КОРОБА В КВ. 21 - РАЗОБРАЛИ, ЖДУТ 16.03.2026 В 10-00",NaN,Принята к исполнению,с 10:00 16.03.2026 по 13:00 16.03.2026,NaN,NaN,ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА,NaN,NaN,0,Нет,Прочее


In [20]:
incidents_json = processor.getIncidentsData()
print(json.dumps(incidents_json["metadata"], ensure_ascii=False, indent=2))
print(json.dumps(incidents_json["incidents"][:3], ensure_ascii=False, indent=2))

{
  "total_incidents": 458
}
[
  {
    "date_start": "15.03.2026",
    "date_end": null,
    "incident_time_start": "23:19",
    "address": "г Санкт-Петербург, п Парголово, ул Николая Рубцова, д. 5 стр. 1",
    "incident_location": "309",
    "subcategory": "Протечка",
    "incident_type": "Утечка стояка",
    "performers": null,
    "materials": null,
    "cost": 0
  },
  {
    "date_start": "15.03.2026",
    "date_end": null,
    "incident_time_start": "21:32",
    "address": "г Санкт-Петербург, пр-кт Гладышевский, д. 38 корп. 2 стр. 1",
    "incident_location": "267",
    "subcategory": "Протечка",
    "incident_type": "Прочее",
    "performers": null,
    "materials": null,
    "cost": 0
  },
  {
    "date_start": "15.03.2026",
    "date_end": null,
    "incident_time_start": "20:48",
    "address": "г Санкт-Петербург, п Парголово, ул Фёдора Абрамова, д. 21 корп. 1 лит. А",
    "incident_location": "575",
    "subcategory": "Протечка",
    "incident_type": "Прочее",
    "performers

In [21]:
processor.getOverview()

,incident,count
0,Замена ХВС на уровне техэтажа,0
1,Замена главных стояков ГВС,0
2,Замена трубы ГВС,6
3,Замена трубы ХВС,0
4,Замена розлива,10
5,Анализ труб,2
6,Утечка радиатора,8
7,Утечка стояка,79
8,Прочее,353
9,Не определен,0
